# Credit Risk Assessment — Retail Bank (Canada)

**Business question:** How can we segment loan applicants by risk profile to reduce default rates and optimize approval decisions?

This notebook loads the IBM credit risk dataset, cleans it, builds a transparent 0–100 risk score,
answers the business question with charts and tables, and exports the numbers used by the dashboard.

In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
OUT_CHARTS = ROOT / "outputs" / "charts"
OUT_DOCS = ROOT / "docs" / "data"
OUT = ROOT / "outputs"
for p in (OUT_CHARTS, OUT_DOCS, OUT):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

## 1. Load the data
The dataset is a public extract of the German Credit data distributed by IBM (credit_risk.csv).

In [2]:
df = pd.read_csv(DATA / "credit_risk.csv")
print(df.shape)
df.head()

(5000, 21)


,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,...,unknown,57,none,own,2,skilled,1,none,yes,Risk


## 2. Clean the data
- Standardise column names and category labels (lowercase, trimmed).
- Remove duplicates and impossible values (age < 18, loan amount <= 0, duration <= 0).
- Create a binary `is_default` flag from `Risk`.

In [3]:
df.columns = [c.strip() for c in df.columns]
cat_cols = df.select_dtypes(include="object").columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip().str.lower()

before = len(df)
df = df.drop_duplicates()
df = df[(df["Age"] >= 18) & (df["LoanAmount"] > 0) & (df["LoanDuration"] > 0)]
df["is_default"] = (df["Risk"] == "risk").astype(int)
print(f"Rows before: {before} | after cleaning: {len(df)} | removed: {before - len(df)}")

Rows before: 5000 | after cleaning: 4982 | removed: 18


## 3. Build a transparent risk score (0–100)
The score combines five drivers weighted by their observed association with default:
checking status (25), credit history (25), loan duration (20), installment burden (15) and savings (15).

In [4]:
checking_map = {"no_checking": 25, "less_0": 20, "0_to_200": 10, "greater_200": 0}
history_map = {"no_credits": 25, "delayed": 20, "critical": 20,
               "prior_payments_delayed": 12, "credits_paid_to_date": 0}
savings_map = {"less_100": 15, "unknown": 12, "100_to_500": 8, "500_to_1000": 4, "greater_1000": 0}

def duration_points(d):
    if d > 36: return 20
    if d > 24: return 12
    if d > 12: return 5
    return 0

def installment_points(i):
    if i >= 4: return 15
    if i == 3: return 8
    return 0

df["risk_score"] = (
    df["CheckingStatus"].map(checking_map).fillna(12)
    + df["CreditHistory"].map(history_map).fillna(10)
    + df["LoanDuration"].apply(duration_points)
    + df["InstallmentPercent"].apply(installment_points)
    + df["ExistingSavings"].map(savings_map).fillna(8)
).clip(0, 100)

bins = [-1, 19, 39, 59, 79, 100]
labels = ["Very Low (0-19)", "Low (20-39)", "Medium (40-59)", "High (60-79)", "Very High (80-100)"]
df["risk_band"] = pd.cut(df["risk_score"], bins=bins, labels=labels)
df["decision"] = np.where(df["risk_score"] < 60, "Approve", "Decline")
print(df[["risk_score", "risk_band", "decision"]].head())

   risk_score       risk_band decision
0        38.0     Low (20-39)  Approve
1        40.0  Medium (40-59)  Approve
2        52.0  Medium (40-59)  Approve
3        45.0  Medium (40-59)  Approve
4        61.0    High (60-79)  Decline


## 4. Key findings
### 4.1 Default rate by risk band
The score is monotonic: default rate rises sharply with the band, confirming the score separates good and bad applicants.

In [5]:
band = (df.groupby("risk_band", observed=True)
          .agg(applicants=("is_default", "size"),
               defaults=("is_default", "sum"),
               default_rate=("is_default", "mean"),
               avg_loan=("LoanAmount", "mean"))
          .reset_index())
band["default_rate_pct"] = (band["default_rate"] * 100).round(2)
band["avg_loan"] = band["avg_loan"].round(0)
print(band[["risk_band", "applicants", "defaults", "default_rate_pct", "avg_loan"]])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(band["risk_band"].astype(str), band["default_rate_pct"], color="#c0392b")
ax.set_title("Default rate by risk band")
ax.set_ylabel("Default rate (%)")
ax.set_xlabel("Risk band")
for i, v in enumerate(band["default_rate_pct"]):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT_CHARTS / "default_rate_by_risk_band.png")
plt.close()

            risk_band  applicants  defaults  default_rate_pct  avg_loan
0     Very Low (0-19)          71        12             16.90    2267.0
1         Low (20-39)        1290       244             18.91    2110.0
2      Medium (40-59)        2005       481             23.99    2745.0
3        High (60-79)        1524       857             56.23    5498.0
4  Very High (80-100)          92        76             82.61    6851.0


### 4.2 Default rate by loan purpose
Certain purposes (e.g. business, education) carry materially higher default rates and should attract tighter rules.

In [6]:
purpose = (df.groupby("LoanPurpose")
             .agg(applicants=("is_default", "size"),
                  defaults=("is_default", "sum"),
                  default_rate=("is_default", "mean"),
                  avg_loan=("LoanAmount", "mean"),
                  exposure=("LoanAmount", "sum"))
             .reset_index())
purpose["default_rate_pct"] = (purpose["default_rate"] * 100).round(2)
purpose["avg_loan"] = purpose["avg_loan"].round(0)
purpose = purpose.sort_values("default_rate_pct", ascending=False)
print(purpose[["LoanPurpose", "applicants", "default_rate_pct", "avg_loan"]])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(purpose["LoanPurpose"], purpose["default_rate_pct"], color="#2c3e50")
ax.invert_yaxis()
ax.set_title("Default rate by loan purpose")
ax.set_xlabel("Default rate (%)")
plt.tight_layout()
plt.savefig(OUT_CHARTS / "default_rate_by_purpose.png")
plt.close()

   LoanPurpose  applicants  default_rate_pct  avg_loan
6        other         113             64.60    6689.0
8      repairs         283             56.54    5006.0
0   appliances         561             51.16    4951.0
7     radio_tv         755             43.18    4120.0
4    education         167             41.92    4029.0
1     business         146             35.62    5068.0
5    furniture         853             33.29    3330.0
10    vacation         205             28.29    3691.0
9   retraining         164             25.61    3874.0
3     car_used         808             24.75    2801.0
2      car_new         927             12.73    1538.0


### 4.3 Default rate by checking status
Checking status is the single strongest liquidity signal: applicants with no checking account default far more often.

In [7]:
checking = (df.groupby("CheckingStatus")
              .agg(applicants=("is_default", "size"),
                   default_rate=("is_default", "mean"))
              .reset_index())
checking["default_rate_pct"] = (checking["default_rate"] * 100).round(2)
checking = checking.sort_values("default_rate_pct", ascending=False)
print(checking)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(checking["CheckingStatus"], checking["default_rate_pct"], color="#8e44ad")
ax.set_title("Default rate by checking account status")
ax.set_ylabel("Default rate (%)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT_CHARTS / "default_rate_by_checking.png")
plt.close()

  CheckingStatus  applicants  default_rate  default_rate_pct
3    no_checking        1993      0.566483             56.65
1    greater_200         305      0.416393             41.64
0       0_to_200        1304      0.217791             21.78
2         less_0        1380      0.094203              9.42


### 4.4 Approval policy simulation
Approving only applicants with a risk score below 60 removes most defaults while keeping a large share of the book.

In [8]:
total = len(df)
current_defaults = int(df["is_default"].sum())
approved = df[df["decision"] == "Approve"]
policy = pd.DataFrame([
    {"policy": "Approve all (current)", "approved": total,
     "approval_rate_pct": 100.0, "defaults": current_defaults,
     "default_rate_pct": round(100 * current_defaults / total, 2)},
    {"policy": "Approve risk score < 60", "approved": len(approved),
     "approval_rate_pct": round(100 * len(approved) / total, 2),
     "defaults": int(approved["is_default"].sum()),
     "default_rate_pct": round(100 * approved["is_default"].mean(), 2)},
])
print(policy)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(policy))
ax.bar(x - 0.2, policy["approval_rate_pct"], width=0.4, label="Approval rate %", color="#27ae60")
ax.bar(x + 0.2, policy["default_rate_pct"], width=0.4, label="Default rate %", color="#c0392b")
ax.set_xticks(x)
ax.set_xticklabels(policy["policy"])
ax.set_title("Approval policy simulation")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_CHARTS / "approval_policy_simulation.png")
plt.close()

                    policy  approved  approval_rate_pct  defaults  \
0    Approve all (current)      4982             100.00      1670   
1  Approve risk score < 60      3366              67.56       737   

   default_rate_pct  
0             33.52  
1             21.90  


### 4.5 Portfolio at risk by purpose
Portfolio at risk (PAR) shows where default exposure is concentrated in dollar terms.

In [9]:
par = purpose[["LoanPurpose", "exposure"]].copy()
par["exposure_at_risk"] = (df[df["is_default"] == 1].groupby("LoanPurpose")["LoanAmount"].sum()
                           .reindex(par["LoanPurpose"]).fillna(0).values)
par["par_pct"] = (100 * par["exposure_at_risk"] / par["exposure"]).round(2)
par = par.sort_values("exposure_at_risk", ascending=False)
print(par)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(par["LoanPurpose"], par["exposure_at_risk"], color="#e67e22")
ax.invert_yaxis()
ax.set_title("Portfolio at risk by loan purpose (exposure)")
ax.set_xlabel("Exposure at risk")
plt.tight_layout()
plt.savefig(OUT_CHARTS / "portfolio_at_risk.png")
plt.close()

   LoanPurpose  exposure  exposure_at_risk  par_pct
0   appliances   2777559           1643416    59.17
7     radio_tv   3110972           1629421    52.38
5    furniture   2840910           1295073    45.59
8      repairs   1416662            973396    68.71
3     car_used   2263518            831744    36.75
6        other    755872            522487    69.12
4    education    672772            401772    59.72
10    vacation    756679            342722    45.29
2      car_new   1425928            303165    21.26
1     business    739996            298722    40.37
9   retraining    635357            211594    33.30


## 5. Export summary numbers and dashboard data

In [10]:
summary = {
    "total_applicants": int(total),
    "default_rate_pct": round(100 * df["is_default"].mean(), 2),
    "approval_rate_pct": round(100 * len(approved) / total, 2),
    "avg_loan_amount": round(float(df["LoanAmount"].mean()), 0),
    "avg_risk_score": round(float(df["risk_score"].mean()), 1),
    "portfolio_at_risk_pct": round(100 * df[df["is_default"] == 1]["LoanAmount"].sum() / df["LoanAmount"].sum(), 2),
    "defaults_avoided": int(current_defaults - approved["is_default"].sum()),
    "high_risk_share_pct": round(100 * (df["risk_score"] >= 60).mean(), 2),
}
with open(OUT / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(summary)

# Dashboard data files (list of records)
band[["risk_band", "applicants", "defaults", "default_rate_pct", "avg_loan"]].to_json(
    OUT_DOCS / "default_by_risk_band.json", orient="records")
purpose[["LoanPurpose", "applicants", "default_rate_pct", "avg_loan"]].to_json(
    OUT_DOCS / "default_by_purpose.json", orient="records")
checking[["CheckingStatus", "applicants", "default_rate_pct"]].to_json(
    OUT_DOCS / "default_by_checking.json", orient="records")
policy.to_json(OUT_DOCS / "approval_policy.json", orient="records")
par[["LoanPurpose", "exposure_at_risk", "par_pct"]].to_json(
    OUT_DOCS / "portfolio_at_risk.json", orient="records")

print("Charts:", sorted(p.name for p in OUT_CHARTS.glob("*.png")))
print("Dashboard data:", sorted(p.name for p in OUT_DOCS.glob("*.json")))

{'total_applicants': 4982, 'default_rate_pct': np.float64(33.52), 'approval_rate_pct': 67.56, 'avg_loan_amount': 3492.0, 'avg_risk_score': 50.3, 'portfolio_at_risk_pct': np.float64(48.59), 'defaults_avoided': 933, 'high_risk_share_pct': np.float64(32.44)}
Charts: ['approval_policy_simulation.png', 'default_rate_by_checking.png', 'default_rate_by_purpose.png', 'default_rate_by_risk_band.png', 'portfolio_at_risk.png']
Dashboard data: ['approval_policy.json', 'default_by_checking.json', 'default_by_purpose.json', 'default_by_risk_band.json', 'portfolio_at_risk.json']
